In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

OUTPUT_DIR = PROJECT_DIR / "OUTPUT"
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
panel = pd.read_csv(CLEAN_DIR / "panel_districts.csv", dtype={"district_key": str, "state_key": str})
print(panel.shape)

(3990, 16)


In [4]:
national = panel.groupby("year")[["bev", "cars_total"]].sum()
national["bev_share_pct"] = national["bev"] / national["cars_total"] * 100

by_region = panel.groupby(["year", "region"])[["bev", "cars_total"]].sum()
by_region["bev_share_pct"] = by_region["bev"] / by_region["cars_total"] * 100
by_region = by_region["bev_share_pct"].unstack()

trend = by_region.copy()
trend["Germany"] = national["bev_share_pct"]
trend.round(2)

region,East,West,Germany
year,,,
2017,0.05,0.08,0.07
2018,0.08,0.12,0.12
2019,0.12,0.19,0.18
2020,0.20,0.30,0.29
2021,0.42,0.68,0.64
2022,0.81,1.37,1.27
2023,1.26,2.24,2.08
2024,1.76,3.09,2.87
2025,2.06,3.60,3.35


In [5]:
correlation = []

for year in range(2022, 2027):
    data = panel[panel["year"] == year]
    correlation.append({
        "year": year,
        "pearson_residence": data["income_median_residence"].corr(data["bev_share_pct"]),
        "spearman_residence": data["income_median_residence"].corr(data["bev_share_pct"], method="spearman"),
        "pearson_workplace": data["income_median_workplace"].corr(data["bev_share_pct"]),
    })

correlation = pd.DataFrame(correlation)
correlation.round(2)

,year,pearson_residence,spearman_residence,pearson_workplace
0,2022,0.64,0.66,0.65
1,2023,0.65,0.65,0.61
2,2024,0.66,0.66,0.64
3,2025,0.74,0.69,0.71
4,2026,0.72,0.68,0.70


In [6]:
recent = panel[panel["year"] >= 2022].copy()

labels = ["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"]
recent["income_group"] = recent.groupby("year")["income_median_residence"].transform(
    lambda s: pd.qcut(s, 5, labels=labels)
)

groups = recent.groupby(["year", "income_group"], observed=True)[["bev", "cars_total"]].sum()
groups["bev_share_pct"] = groups["bev"] / groups["cars_total"] * 100
gap = groups["bev_share_pct"].unstack()

gap["Q5 - Q1"] = gap["Q5 highest"] - gap["Q1 lowest"]
gap["Q5 / Q1"] = gap["Q5 highest"] / gap["Q1 lowest"]
gap.round(2)

income_group,Q1 lowest,Q2,Q3,Q4,Q5 highest,Q5 - Q1,Q5 / Q1
year,,,,,,,
2022,0.71,1.17,1.22,1.26,1.65,0.94,2.33
2023,1.16,1.95,2.04,2.07,2.60,1.44,2.24
2024,1.59,2.67,2.73,2.85,3.66,2.07,2.30
2025,1.84,3.14,3.22,3.32,4.25,2.41,2.31
2026,2.28,3.90,3.88,4.11,5.20,2.92,2.28


In [7]:
latest = panel[panel["year"] == 2026]

for region in ["West", "East"]:
    data = latest[latest["region"] == region]
    r = data["income_median_residence"].corr(data["bev_share_pct"])
    print(f"{region}: {len(data)} districts, Pearson r = {r:.2f}")

West: 323 districts, Pearson r = 0.54
East: 76 districts, Pearson r = 0.84


In [8]:
latest = panel[panel["year"] == 2026]
THRESHOLD = 750

outliers = latest[latest["cars_per_1000_inh"] > THRESHOLD]
print(outliers[["district_name", "cars_per_1000_inh", "bev_share_pct"]]
      .sort_values("cars_per_1000_inh", ascending=False).round(1))

without = latest[latest["cars_per_1000_inh"] <= THRESHOLD]

for label, data in [("All districts", latest), ("Without outliers", without)]:
    pearson = data["income_median_residence"].corr(data["bev_share_pct"])
    spearman = data["income_median_residence"].corr(data["bev_share_pct"], method="spearman")
    print(f"{label:17} n={len(data)}  Pearson={pearson:.2f}  Spearman={spearman:.2f}")

                                      district_name  cars_per_1000_inh  \
189                     Wolfsburg, kreisfreie Stadt              975.6   
849                               Euskirchen, Kreis              833.7   
1199  Wiesbaden, kreisfreie Stadt, Landeshauptstadt              772.8   
1259                              Main-Taunus-Kreis              770.3   
1859                      Hohenlohekreis, Landkreis              766.6   
1769                        Südwestpfalz, Landkreis              765.3   

      bev_share_pct  
189            12.5  
849             7.0  
1199            8.1  
1259            4.1  
1859            5.8  
1769            3.1  
All districts     n=399  Pearson=0.72  Spearman=0.68
Without outliers  n=393  Pearson=0.75  Spearman=0.68


In [9]:
variables = ["bev_share_pct", "income_median_residence", "charging_points_per_100k_inh"]

latest[variables].corr(method="spearman").round(2)

,bev_share_pct,income_median_residence,charging_points_per_100k_inh
bev_share_pct,1.00,0.68,0.48
income_median_residence,0.68,1.00,0.25
charging_points_per_100k_inh,0.48,0.25,1.00


In [10]:
trend.to_csv(OUTPUT_DIR / "bev_share_trend.csv")
correlation.to_csv(OUTPUT_DIR / "correlation_by_year.csv", index=False)
gap.to_csv(OUTPUT_DIR / "bev_share_by_income_group.csv")

print("Saved files:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" ", file.name)

Saved files:
  bev_share_by_income_group.csv
  bev_share_trend.csv
  correlation_by_year.csv
